Solve the equation
$$
y(x) - \sin{x} = 0
$$
using pytorch tools. 

In [ ]:
# it's always good to...

%reset -f

In [ ]:
# pytorch is an open source machine learning library

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time

1. Define the neural network.

We will use a standard <b>Feedforward Neural Network</b> (FNN) also known as a <b> MultiLayer Perceptron </b>(MLP).

Input: scalar $x$.  Output: scalar $y(x)$. 

$$
y(x) \simeq {\rm NN}(x;\theta)
$$

where $\theta$ are the trainable weights. 

Hidden layers use the $\tanh$ activation function, which is smooth and good at modeling oscillations.

In [ ]:
### Define the neural network ###

class my_NN(nn.Module):
    
    def __init__(self, layers):
        super(my_NN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
        self.activation = torch.tanh

    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = self.activation(self.layers[i](x))
        return self.layers[-1](x)

2. Set up the problem

For this simple example, all we need to do is define the domain of $x$ and the number of <b>collocation points</b> at which we want to enforce the solution. 

In [ ]:
### set up the problem ###

# conditions used in neuronz.ipynb

xmin =  -2.0*np.pi      # lower limit of domain
xmax =   2.0*np.pi      # upper limit of domain 
N_collocation = 200     # points at which loss functions will be evaluated

x_col = torch.linspace(xmin,xmax,N_collocation).view(-1,1) # collocation points as column vector

3. Loss Function:

a) Predict solution at collocation points: $y(x) = {\rm NN}(x,\theta)$

b) Build the residual (function to be zeroed): $R(x) = y(x) - \sin{x}$

c) Define the loss function. Pretty simple in this case: ${\cal L} = R^2(x)$

In [ ]:
### Loss function ###

def mynn_loss(model):
    
    # predict solution at collocation points
    y = model(x_col)
    
    # build residual     
    R = y - torch.sin(x_col) 
    
    # define loss   
    loss = torch.mean(R**2)

    return loss

4. Instantiate and Train Model:

a) Define the layers of the NN

b) Choose the optimizing algorithm:

* <b>Stochastic Gradient Descent</b> as used in neuronz.ipynb

* The <b>Adam</b> optimizing algorithm  (Kingma and Ba, https://arxiv.org/abs/1412.6980) <br>  <i>n.b. Gradient descent takes a step in the downhill direction using just the current slope and one fixed learning rate for every weight. Adam adds two strategies to make that walk smoother and better scaled: (1) it remembers recent directions (so noisy steps average out and progress builds along steady slopes) and (2) adjusts step size per weight (taking smaller steps where slopes are steep/jumpy and bigger steps where slopes are gentle/steady).</i>

c) Loop through a fixed number of training <b>epochs</b> to minimize the loss function

In [ ]:
### Instantiate the neural network and choose optimizer routine ###

# create the network. compare neuronz.ipynb

layers = [1, 2, 1] 
mynn   = my_NN(layers)

# choose optimizer. stochastic gradient descent (SGD) as in neuronz.ipynb 

optimizer = torch.optim.SGD(mynn.parameters(), lr=0.01)

In [ ]:
### Train the neural network ###

print("")
print("Training the Neural Network...")
print("")

start_train = time.time()
loss_history = []

for epoch in range(5000):
    
    optimizer.zero_grad()
    loss = mynn_loss(mynn)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    
    if epoch % 500 == 0:
        print("Epoch %5d, Loss = %.4e" % (epoch,loss.item()))

print("")
end_train = time.time()

In [ ]:
print("training runtime: %7.4f seconds" % (end_train-start_train))

5: Plot solution and diagnostics

In [ ]:
### plot solution ###

x_test   = np.linspace(xmin,xmax,1000)
x_tensor = torch.tensor(x_test,dtype=torch.float32).view(-1, 1)

# NN result
with torch.no_grad():
    y_pred = mynn(x_tensor)

# analytic solution
y_true  = torch.sin(x_tensor)

plt.figure(figsize=(8, 5))
plt.plot(x_test, y_true, 'k--', label="exact solution")
plt.plot(x_test, y_pred, 'r-',  label="my_NN prediction")
plt.xlabel("x")
plt.ylabel("y(x)")
plt.legend()
plt.title("Sinusoid via my_NN")
plt.savefig("sinusoid.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### plot residual ###

plt.figure(figsize=(8, 5))

plt.plot(x_test, y_pred - y_true, 'k--', label="solution error")
plt.xlabel("x")
plt.ylabel("$y_{mynn} - \sin{x}$")
plt.legend()
plt.title("Sinusoid via my_NN")
plt.savefig("sinusoid_error.png",dpi=300,facecolor='white', transparent=False)
plt.show()

In [ ]:
### plot training loss history ###

plt.figure(figsize=(8, 5))
plt.semilogy(loss_history, 'r-')
plt.xlabel("Epoch")
plt.ylabel("Loss (log scale)")
plt.title("Training Loss History")
plt.savefig("sinusoid_loss_history.png",dpi=300,facecolor='white', transparent=False)
plt.show()